# Case 1 — TransformerDịch máy **Anh → Việt** trên IWSLT'15 — Transformer encoder-decoder (Vaswani et al., 2017)> Trước khi chạy: **Runtime → Change runtime type → T4 GPU**.>> Toàn bộ notebook mất khoảng **1.5–2.5 giờ** trên T4 miễn phí.

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smiimport torchassert torch.cuda.is_available(), "Chưa bật GPU! Runtime -> Change runtime type -> T4 GPU"print(f"\nGPU     : {torch.cuda.get_device_name(0)}")print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")print(f"PyTorch : {torch.__version__}")

## 2. Lấy code từ GitHubSửa `REPO` thành repo của bạn rồi chạy. Ô này dùng được cả lần đầu (clone) lẫnnhững lần sau (pull code mới nhất), nên mỗi khi bạn sửa code ở máy và push lênthì chỉ cần chạy lại ô này.`git reset --hard` chỉ ghi đè file đã commit. Thư mục `runs/` nằm trong`.gitignore` nên checkpoint và kết quả train vẫn còn nguyên.

In [ ]:
# ============ SỬA DÒNG NÀY ============REPO   = "https://github.com/<user>/<repo>.git"BRANCH = "main"# ======================================import pathlibNAME = REPO.rstrip("/").split("/")[-1].replace(".git", "")ROOT = pathlib.Path("/content") / NAMEif (ROOT / ".git").exists():    print(f"Đã có {ROOT}, lấy code mới nhất")    !git -C {ROOT} fetch -q origin {BRANCH}    !git -C {ROOT} reset -q --hard origin/{BRANCH}else:    !git clone -q -b {BRANCH} {REPO} {ROOT}%cd {ROOT}!git log -1 --format="commit %h  |  %s  |  %cr"print()!ls# Repo private thì dùng token thay cho dòng REPO ở trên:#   REPO = "https://<TOKEN>@github.com/<user>/<repo>.git"## Chưa push lên GitHub thì upload zip thay thế:#   from google.colab import files; import zipfile#   up = files.upload()#   zipfile.ZipFile(list(up)[0]).extractall("/content")#   %cd /content/machine_translate

## 3. Cài thư việnColab đã có sẵn PyTorch; chỉ cần thêm hai package nhẹ.

In [ ]:
!pip install -q sentencepiece 'sacrebleu>=2.4'print("xong")

## 4. Tải dữ liệu IWSLT'15 En-ViScript tự verify đủ số câu (133,317 / 1,553 / 1,268) và dừng ngay nếu lệch.

In [ ]:
!bash scripts/download_data.sh

## 5. Train tokenizer dùng chung**Chỉ chạy một lần** và phải chạy trước cả hai case — cả hai đều nạp đúng bộtokenizer này, đó là điều kiện để so sánh công bằng. Nếu bạn đã chạy notebookcủa case kia trong cùng phiên Colab thì bỏ qua ô này.

In [ ]:
!python scripts/prepare.py

## 6. Kiểm tra trước khi trainHai bước này rẻ và bắt được lỗi trước khi bạn đốt hàng giờ GPU.- `check_amp.py` — chạy cả hai model dưới fp16. Lỗi lệch kiểu fp16/fp32 vô hình  trên CPU nhưng làm hỏng ngay lần train đầu trên GPU.- `sanity_check.py` — bắt model học thuộc 120 câu; cài đặt đúng phải đạt BLEU > 80.

In [ ]:
!python scripts/check_amp.py

In [ ]:
!python scripts/sanity_check.py --model transformer

## 7. Train TransformerCấu hình mặc định (`translate_transformers/config.py`): 6+6 lớp, d_model=512,4 head, d_ff=1024, **dropout 0.3**, pre-norm, weight tying, fp16, batch gom theotoken (8192), inverse-sqrt LR với 4000 bước warmup, tối đa 30 epoch, early stopkhi BLEU dev đứng yên 5 epoch.Ước tính **2–4 phút/epoch** trên T4. BLEU dev được chấm sau mỗi epoch nên theodõi được ngay.**Nếu Colab ngắt phiên giữa chừng:** chạy lại ô này với `--resume true`. Stateđầy đủ (model, optimizer, scheduler, history) được ghi vào `runs/transformer/last.pt`sau mỗi epoch.

In [ ]:
!python translate_transformers/train.py# Train tiếp sau khi bị ngắt:# !python translate_transformers/train.py --resume true# Tinh chỉnh:# !python translate_transformers/train.py --epochs 40 --dropout 0.2 --max-tokens 6144

## 8. Đánh giá với beam size khác`train.py` đã chấm sẵn greedy và beam mặc định. Ô này để thử beam khác hoặc xemthêm câu dịch mẫu mà không phải train lại.

In [ ]:
!python evaluate.py --model transformer --beam 5 --show 8

## 9. Dịch thử câu bất kỳ

In [ ]:
!python evaluate.py --model transformer --beam 5 \    --text "I want to talk about the science behind climate change ."

## 10. So sánh hai caseÔ này chỉ ra kết quả đầy đủ khi **cả hai** case đã train xong trong cùng phiênColab (hoặc bạn đã copy `runs/` của case kia vào).

In [ ]:
!python compare.py --plots

In [ ]:
from IPython.display import Image, Markdown, displayimport osif os.path.exists('runs/comparison.png'):    display(Image('runs/comparison.png'))if os.path.exists('runs/COMPARISON.md'):    display(Markdown(open('runs/COMPARISON.md').read()))

## 11. Tải kết quả về máy`runs/` gồm `benchmark.json` (mọi số đo), `history.csv` (learning curve), câudịch trên tst2013 và checkpoint. Tải về trước khi phiên Colab hết hạn.

In [ ]:
!zip -qr runs.zip runs -x '*/last.pt'from google.colab import filesfiles.download('runs.zip')